# Notebook 04 — Оценка качества модели

## Что делаем:
1. Загружаем обученную модель (best_model.pth)
2. Вычисляем метрики на тестовой выборке: MSE, MAE, R²
3. Строим график предсказания vs реальность
4. Анализируем распределение ошибок
5. Показываем примеры удачных и неудачных предсказаний

In [ ]:
!pip install -q torch torchvision numpy pandas matplotlib opencv-python scikit-learn tqdm

from google.colab import drive
drive.mount('/content/drive')

import sys, os, json
import torch
import numpy as np
import matplotlib
import matplotlib.pyplot as plt
matplotlib.rcParams['figure.dpi'] = 120

with open('/content/drive/MyDrive/diploma/config.json') as f:
    cfg = json.load(f)

PROJECT_DIR = cfg['project_dir']
MODELS_DIR  = cfg['models_dir']
OUTPUTS_DIR = cfg['outputs_dir']
CSV_PATH    = cfg['csv_path']

src_path = os.path.join(PROJECT_DIR, 'src')
if src_path not in sys.path:
    sys.path.insert(0, src_path)

torch.manual_seed(42)
np.random.seed(42)

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Устройство: {device}')

## Шаг 1. Загрузка модели

In [ ]:
from model import get_model

model_path = os.path.join(MODELS_DIR, 'best_model.pth')

if not os.path.exists(model_path):
    raise FileNotFoundError(f'Модель не найдена: {model_path}\nСначала запусти notebook 03!')

model = get_model(dropout_rate=0.5, device=device)
model.load_state_dict(torch.load(model_path, map_location=device))
model.eval()

print(f'✓ Модель загружена: {model_path}')
size_mb = os.path.getsize(model_path) / 1e6
print(f'  Размер: {size_mb:.1f} MB')

## Шаг 2. Тестовые данные

In [ ]:
from dataset import get_dataloaders

_, _, test_loader = get_dataloaders(
    csv_path=CSV_PATH,
    data_dir=os.path.dirname(CSV_PATH),
    batch_size=32,
    balance=True,
    num_workers=0,
)

print(f'Тестовых батчей: {len(test_loader)}')
print(f'Примерно сэмплов: {len(test_loader) * 32}')

## Шаг 3. Метрики качества

| Метрика | Описание | Идеал |
|---------|----------|-------|
| MSE | Среднеквадратичная ошибка | → 0 |
| RMSE | Корень из MSE (в тех же единицах что угол) | → 0 |
| MAE | Средняя абсолютная ошибка | → 0 |
| R² | Доля объяснённой дисперсии | → 1 |

In [ ]:
from evaluate import get_predictions, compute_metrics

y_true, y_pred = get_predictions(model, test_loader, device)
metrics = compute_metrics(y_true, y_pred)

## Шаг 4. Визуализация: предсказания vs реальность

In [ ]:
from evaluate import plot_predictions_vs_actual

plot_predictions_vs_actual(
    y_true, y_pred,
    save_path=os.path.join(OUTPUTS_DIR, 'pred_vs_actual.png')
)

## Шаг 5. Распределение ошибок

In [ ]:
from evaluate import plot_error_distribution

plot_error_distribution(
    y_true, y_pred,
    save_path=os.path.join(OUTPUTS_DIR, 'error_distribution.png')
)

## Шаг 6. Сетка 3×3 — примеры предсказаний на изображениях

In [ ]:
from evaluate import plot_sample_predictions

plot_sample_predictions(
    model, test_loader, device,
    save_path=os.path.join(OUTPUTS_DIR, 'sample_predictions.png')
)

## Шаг 7. Анализ ошибок по диапазонам углов

In [ ]:
# Разбиваем на группы по типу манёвра и считаем MAE для каждой
from sklearn.metrics import mean_absolute_error

groups = {
    'Прямо (|угол| < 0.05)':      np.abs(y_true) < 0.05,
    'Лёгкий поворот (0.05–0.2)':  (np.abs(y_true) >= 0.05)  & (np.abs(y_true) < 0.2),
    'Сильный поворот (> 0.2)':    np.abs(y_true) >= 0.2,
}

print('MAE по типам манёвров:')
print('-' * 45)
for name, mask in groups.items():
    if mask.sum() > 0:
        mae_g = mean_absolute_error(y_true[mask], y_pred[mask])
        count = mask.sum()
        print(f'  {name:<35} MAE={mae_g:.4f}  n={count}')
print('-' * 45)

# Визуализация
fig, ax = plt.subplots(figsize=(10, 4))

# Scatter, окрашенный по ошибке
errors = np.abs(y_pred - y_true)
sc = ax.scatter(y_true, y_pred, c=errors, cmap='RdYlGn_r', s=8, alpha=0.5,
                vmin=0, vmax=0.3)
plt.colorbar(sc, ax=ax, label='Абс. ошибка')

lim = 1.1
ax.plot([-lim, lim], [-lim, lim], 'k--', linewidth=1.5, label='Идеал')
ax.set_xlabel('Реальный угол', fontsize=12)
ax.set_ylabel('Предсказанный угол', fontsize=12)
ax.set_title('Качество предсказаний (цвет = ошибка)', fontsize=12, fontweight='bold')
ax.legend(); ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig(os.path.join(OUTPUTS_DIR, 'error_by_angle.png'), dpi=150)
plt.show()

## Итог

✅ Метрики вычислены (MSE, MAE, R²)  
✅ Графики предсказаний построены  
✅ Проведён анализ ошибок по типам манёвров  
✅ Все визуализации сохранены в `outputs/`  

**Следующий шаг:** `05_yolo_detection.ipynb` — детекция объектов на дороге (YOLOv8)